# 🌲 Step 0 — Smart Upload & Preview
**Forest Carbon Continuity Pipeline · Sentinel-1/2 Hybrid for EU LULUCF Compliance**

## What this step does
Validates and previews your **National Forest Inventory (NFI) ground-truth data** before any
processing happens. It generates a realistic synthetic NFI table so the notebook runs standalone
on day one, gives you a blank template to fill in, and lets you upload your own CSV/XLSX —
your file replaces the synthetic table everywhere below without changing a line of logic.

## The Science
NFI plots record DBH (diameter at breast height, measured at 1.3 m), species, and hypsometer
sighting fields (slope `Distance`, `Angle`, observer `ObsHeight`). EU LULUCF rules only count
**trees with DBH ≥ 10 cm** — anything thinner is not "forest" for carbon accounting. Letting bad
or thin plots slip through poisons every downstream phase, so this step is the quality gate.

In [ ]:
# ============================================================
# Environment detection & package setup (runs first, auto-detects
# Google Colab vs local Jupyter - nothing needs manual pip install)
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

_ensure("numpy"); _ensure("pandas"); _ensure("matplotlib")
_ensure("rasterio"); _ensure("geopandas"); _ensure("scikit-learn")
_ensure("openpyxl"); _ensure("ipywidgets")

def outpath(name):
    """Returns a working save path for the current environment."""
    return f"/content/{name}" if IN_COLAB else name

print(f"Environment detected: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

# ============================================================
# Unified cross-environment helpers:
# download buttons, regenerate buttons, upload widgets
# ============================================================
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

_generated_files = {}  # friendly label -> filepath, used by the final "Download All" bundle

def register_file(label, path):
    if label:
        _generated_files[label] = path

def make_download_button(path, label=None, register_as=None):
    """Cross-environment download button. Colab: real browser download.
    Local Jupyter: clickable FileLink. Reads the file fresh at click time."""
    label = label or f"Download {path}"
    if register_as:
        register_file(register_as, path)
    btn = widgets.Button(description=label, button_style="info", icon="download")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            if IN_COLAB:
                colab_files.download(path)
            else:
                display(FileLink(path))
                print(f"Click the link above to download '{path}'.")

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_regenerate_button(rerun_fn, label="🔄 Regenerate & Re-run"):
    """Button that calls rerun_fn() (fresh random seed, full re-run of this
    section) and shows the refreshed output inline below the button."""
    btn = widgets.Button(description=label, button_style="warning", icon="refresh")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            rerun_fn()

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_upload_widget(on_loaded, accept=".csv,.xlsx", label="Upload replacement file"):
    """Colab: native blocking file picker via google.colab.files.upload().
    Local Jupyter: ipywidgets.FileUpload + Load button (non-blocking)."""
    out = widgets.Output()

    if IN_COLAB:
        btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                print("Choose a file in the dialog, or Cancel to keep current data.")
                uploaded = colab_files.upload()
                if uploaded:
                    fname = next(iter(uploaded))
                    on_loaded(fname, uploaded[fname])
                else:
                    print("No file selected - keeping current data.")

        btn.on_click(on_click)
        display(widgets.VBox([btn, out]))
    else:
        uploader = widgets.FileUpload(accept=accept, multiple=False, description="Choose file")
        load_btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                val = uploader.value
                if not val:
                    print("No file selected yet - choose a file above, then click the button.")
                    return
                item = list(val.values())[0] if isinstance(val, dict) else val[0]
                if isinstance(item, dict) and "metadata" in item:
                    fname = item["metadata"]["name"]     # ipywidgets 7.x shape
                else:
                    fname = item["name"]                  # ipywidgets 8.x shape
                on_loaded(fname, bytes(item["content"]))

        load_btn.on_click(on_click)
        display(widgets.VBox([uploader, load_btn, out]))

def make_download_all_button(label="⬇️ Download ALL outputs (.zip)"):
    """Zips every output registered so far and offers it as one download."""
    zip_path = outpath("outputs_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _label, _path in _generated_files.items():
            try:
                zf.write(_path)
            except FileNotFoundError:
                pass
    make_download_button(zip_path, label=label)

print("Helpers ready: make_download_button / make_regenerate_button / make_upload_widget")

In [ ]:
# ============================================================
# Standalone synthetic NFI data (Day-1 ready, no external files)
# ============================================================
import numpy as np
import pandas as pd

np.random.seed(42)

def generate_synthetic_nfi(n_plots=60, seed=42):
    """Synthetic National Forest Inventory plot table for SE Ireland.
    Columns match the pipeline's NFI contract (see the template below)."""
    rng = np.random.default_rng(seed)
    lon = rng.uniform(-6.40, -6.10, n_plots)
    lat = rng.uniform(52.95, 53.15, n_plots)
    dbh = rng.uniform(4, 45, n_plots)                                # cm - some below the 10 cm EU threshold
    species = rng.choice([1, 2, 3], n_plots, p=[0.55, 0.25, 0.20])   # 1 Sitka Spruce, 2 Lodgepole Pine, 3 Native Mixed
    distance = rng.uniform(5, 25, n_plots)                           # m, hypsometer slope distance
    angle = rng.uniform(15, 55, n_plots)                             # degrees to the prism
    obs_height = np.full(n_plots, 1.5)                               # m, eye height
    return pd.DataFrame({"x": lon.round(6), "y": lat.round(6), "DBH": dbh.round(1),
                         "Species": species, "Distance": distance.round(1),
                         "Angle": angle.round(1), "ObsHeight": obs_height})

nfi_df = generate_synthetic_nfi()
print(f"Synthetic NFI generated: {len(nfi_df)} plots")
nfi_df.head()

## Optional: upload your own NFI data
Everything below runs on the synthetic table by default, so **no upload is required**. If you have
a real NFI export, fill in the template and upload it — the file replaces the synthetic table for
the rest of the notebook.

Required columns and units (one row per plot):

| Column | Units | Description |
|---|---|---|
| `x` | degrees longitude | Plot longitude (WGS84) |
| `y` | degrees latitude | Plot latitude (WGS84) |
| `DBH` | cm | Diameter at breast height (≥ 10 cm passes the EU filter) |
| `Species` | 1 / 2 / 3 | 1 = Sitka Spruce, 2 = Lodgepole Pine, 3 = Native Mixed |
| `Distance` | m | Hypsometer slope distance to the tree |
| `Angle` | degrees | Sighting angle to the prism |
| `ObsHeight` | m | Observer eye height (typically 1.5) |

In [ ]:
# ============================================================
# Optional: download the blank template / upload your own NFI data
# ============================================================
REQUIRED_COLUMNS = ["x", "y", "DBH", "Species", "Distance", "Angle", "ObsHeight"]

template_path = outpath("NFI_template.xlsx")
nfi_df.head(3).to_excel(template_path, index=False)
print("Template pre-filled with the required headers (3 example rows):")
make_download_button(template_path, label="⬇️ Download blank NFI template (.xlsx)",
                     register_as="NFI upload template")

uploaded_nfi_df = None

def handle_nfi_upload(fname, content_bytes):
    global uploaded_nfi_df, nfi_df
    import io
    try:
        candidate = (pd.read_csv(io.BytesIO(content_bytes)) if fname.lower().endswith(".csv")
                     else pd.read_excel(io.BytesIO(content_bytes)))
    except Exception as e:
        print(f"❌ Could not parse '{fname}': {e}")
        return
    missing = [c for c in REQUIRED_COLUMNS if c not in candidate.columns]
    if missing:
        print(f"❌ Upload rejected - missing required column(s): {missing}. Synthetic data stays active.")
        return
    uploaded_nfi_df = candidate.copy()
    nfi_df = uploaded_nfi_df
    print(f"✅ SUCCESS: loaded {len(nfi_df)} plots from '{fname}'. It now replaces the synthetic table below.")

make_upload_widget(handle_nfi_upload, accept=".csv,.xlsx", label="Upload NFI .csv/.xlsx")
print("Skip the upload to continue with the synthetic plots - nothing below requires a file.")

In [ ]:
# ============================================================
# Validation & preview: DBH filter + spatial scatter + histogram
# ============================================================
import matplotlib.pyplot as plt

valid = nfi_df[nfi_df["DBH"] >= 10.0].copy()
rejected = len(nfi_df) - len(valid)
print(f"EU LULUCF threshold DBH >= 10 cm: kept {len(valid)} / {len(nfi_df)} plots ({rejected} rejected)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(nfi_df["x"], nfi_df["y"], c=nfi_df["Species"], cmap="Greens",
                s=28, edgecolor="k", linewidth=0.3)
axes[0].set_title("NFI plot locations (colored by species)")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
axes[1].hist(nfi_df["DBH"], bins=20, color="#2e7d32", edgecolor="white")
axes[1].axvline(10, color="red", linestyle="--", label="10 cm EU threshold")
axes[1].set_title("DBH distribution"); axes[1].set_xlabel("DBH (cm)"); axes[1].legend()
plt.tight_layout()

preview_png = outpath("step0_nfi_preview.png")
fig.savefig(preview_png, dpi=120, bbox_inches="tight")
plt.show()

make_download_button(preview_png, label="⬇️ Download preview chart (.png)",
                     register_as="Step 0 preview chart")

**Interpretation:** the scatter shows plot coverage across the licence area and the histogram
shows how many inventory trees fall below the 10 cm EU threshold. Everything to the **left of the
red dashed line is dropped** from carbon accounting — that is the contract every later phase
inherits. If you uploaded your own file, the exact same filter just ran on your data.

In [ ]:
# ============================================================
# Export validated table + regenerate control
# ============================================================
validated_path = outpath("step0_nfi_validated.xlsx")
valid.to_excel(validated_path, index=False)
make_download_button(validated_path, label="⬇️ Download validated NFI table (.xlsx)",
                     register_as="Step 0 validated NFI")

def rerun_step0(seed=None):
    """Regenerates the synthetic NFI with a fresh seed and re-applies the DBH filter."""
    global nfi_df, valid
    seed = seed or np.random.randint(0, 10_000)
    nfi_df = generate_synthetic_nfi(seed=seed)
    valid = nfi_df[nfi_df["DBH"] >= 10.0].copy()
    print(f"[seed={seed}] regenerated {len(nfi_df)} synthetic plots ({len(valid)} pass the DBH filter)")

make_regenerate_button(rerun_step0, label="🔄 Regenerate synthetic NFI & re-check")
make_download_all_button()